# SLAM3R reconstruction — R1 cell scan

Runs [SLAM3R](https://github.com/PKU-VCL-3DV/SLAM3R) (arXiv 2412.09401) on frames captured from the R1 by `scan_capture.py` and corrected by `undistort.py`. The Mac cannot do this part — SLAM3R needs CUDA.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Free tier is enough; the models run at 224×224.

Expect ~10 min of setup and a few minutes of reconstruction for a 370-frame scan.

## 1 — Confirm a GPU is attached

If this fails, you are on a CPU runtime and nothing below will work.

In [ ]:
!nvidia-smi
import torch, sys
assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> T4 GPU"
print(torch.cuda.get_device_name(0))

## 2 — Install

Torch is pinned to **2.5.0**, the version SLAM3R was written against, on a cu121 build rather than the repo's cu118 because that is what Colab's driver serves. The pin matters for more than tidiness: torch 2.6 flipped `torch.load` to `weights_only=True` by default, which breaks loading the SLAM3R checkpoints.

This downloads a couple of GB and takes several minutes.

In [ ]:
%cd /content
!git clone --depth 1 https://github.com/PKU-VCL-3DV/SLAM3R.git 2>/dev/null || echo "already cloned"
%cd /content/SLAM3R

!pip install -q torch==2.5.0 torchvision==0.20.0 --index-url https://download.pytorch.org/whl/cu121
!pip install -q -r requirements.txt

# xformers 0.0.28.post2 is the build matched to torch 2.5.0. Optional: it only
# makes attention faster, so a failure here is not fatal.
!pip install -q xformers==0.0.28.post2 || echo "xformers skipped"

import torch
print("torch", torch.__version__, "cuda", torch.version.cuda)

In [ ]:
# The RoPE CUDA kernel. Also optional — there is a pure-PyTorch fallback, just
# slower. If this errors, carry on to the next cell.
%cd /content/SLAM3R/slam3r/pos_embed/curope/
!python setup.py build_ext --inplace 2>&1 | tail -5
%cd /content/SLAM3R

## 3 — Upload the scan

Drag **`scans/cell01_undist.zip`** (42 MB) into the file pane on the left, into `/content/`, then run this.

Upload the *undistorted* zip, not the raw frames. SLAM3R's own `docs/recon_tips.md` lists wide-angle lenses and unfamiliar distortion among its known failure modes, and the R1's camera is a 160° fisheye.

In [ ]:
import glob, os, zipfile

ZIP = "/content/cell01_undist.zip"
SCAN = "/content/scan/cell01_undist"

assert os.path.exists(ZIP), "Drag cell01_undist.zip into /content/ in the file pane first"
with zipfile.ZipFile(ZIP) as z:
    z.extractall("/content/scan")

frames = sorted(glob.glob(f"{SCAN}/*.jpg"))
print(f"{len(frames)} frames")

from PIL import Image
print("size", Image.open(frames[0]).size)
Image.open(frames[len(frames) // 2])

## 4 — Reconstruct

Parameters follow `scripts/demo_wild.sh`, which is the configuration tuned for casually captured video rather than for the benchmark datasets. Three are set for *this* scan specifically:

- **`keyframe_stride`** — capture ran at 6 fps with a median frame-to-frame overlap of 0.87, which is dense. Stride 3 spaces keyframes out to roughly 0.6 overlap: enough parallax to triangulate, enough overlap to register. The `keyframe_adapt_*` bounds let SLAM3R override this per window if it disagrees.
- **`buffer_size`** — 370 frames at stride 3 is about 123 keyframes, so 150 holds the whole cell in the retrieval buffer at once. With `reservoir` (the strategy for single-room scenes) that means returning to a spot you already visited can still close the loop.
- **`conf_thres_i2p`** — left at the default. Raise it toward 3.0 if the cloud comes out noisy; the skylights and windows in this cell blow out to white, and the tips doc calls for a higher threshold exactly where depth is unpredictable.

In [ ]:
%cd /content/SLAM3R
!python recon.py \
    --img_dir /content/scan/cell01_undist \
    --test_name cell01 \
    --save_dir /content/results \
    --keyframe_stride 3 \
    --win_r 5 \
    --num_scene_frame 10 \
    --initial_winsize 5 \
    --conf_thres_i2p 1.5 \
    --conf_thres_l2w 12 \
    --max_num_register 10 \
    --num_points_save 1000000 \
    --buffer_size 150 \
    --buffer_strategy reservoir \
    --update_buffer_intv 1 \
    --retrieve_freq 1 \
    --keyframe_adapt_min 1 \
    --keyframe_adapt_max 20 \
    --keyframe_adapt_stride 1 \
    --gpu_id 0

## 5 — Look at the result before downloading

Worth checking here rather than after a download: if the reconstruction failed it usually failed obviously.

In [ ]:
import glob, os

plys = sorted(glob.glob("/content/results/**/*.ply", recursive=True),
              key=os.path.getsize, reverse=True)
for p in plys[:10]:
    print(f"{os.path.getsize(p)/1e6:8.1f} MB  {p}")
assert plys, "no point cloud produced — read the log above"
cloud = plys[0]

In [ ]:
# Three orthogonal projections. A good reconstruction of one cell shows a flat
# floor plane and walls meeting it at right angles; drift shows up as a floor
# that curves or as the same wall appearing twice.
import numpy as np, trimesh, matplotlib.pyplot as plt

pc = trimesh.load(cloud)
xyz = np.asarray(pc.vertices)
rgb = (np.asarray(pc.colors)[:, :3] / 255.0) if pc.colors is not None and len(pc.colors) else None
print(f"{len(xyz):,} points   extent {xyz.ptp(axis=0).round(2)}")

sel = np.random.default_rng(0).choice(len(xyz), min(120_000, len(xyz)), replace=False)
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, (i, j), name in zip(axes, [(0, 2), (0, 1), (2, 1)],
                            ["top (x-z)", "front (x-y)", "side (z-y)"]):
    ax.scatter(xyz[sel, i], xyz[sel, j], s=0.4,
               c=rgb[sel] if rgb is not None else "k", linewidths=0)
    ax.set_title(name); ax.set_aspect("equal"); ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# Download. Open the .ply in MeshLab, CloudCompare, or Blender on the Mac.
from google.colab import files
files.download(cloud)

## If it looks wrong

| Symptom | Try |
|---|---|
| Cloud is a noisy haze, no clear geometry | `--conf_thres_i2p 3.0` — the blown-out windows in this cell are the likeliest culprit |
| Scene tears apart partway through | Lower `--keyframe_stride` to 2. Somewhere the robot moved too fast; `manifest.json` records where overlap dipped (worst frame in this scan is 0.33) |
| Everything registers but is subtly warped | The 160° undistortion estimate is off. Re-run `undistort.py scans/cell01 --auto` on the Mac, or `--tune` to set it by eye |
| Same wall appears twice, offset | Accumulated drift. The paper notes there is no global bundle adjustment. Rescan with a deliberate loop back to the start |
| Only part of the cell reconstructed | Raise `--buffer_size`, or switch `--buffer_strategy fifo` if you scanned more than one room |
| CUDA out of memory | Lower `--win_r` to 3 and `--num_scene_frame` to 5 |

Colab disconnects on idle and wipes `/content`. If you plan to iterate on parameters, mount Drive and point `--save_dir` at it so results survive.